# Practical P21: LlamaIndex Document Search App & RAG Architecture
**Course**: PGDCA — Hands-On Large Language Models  
**Unit 3**: LLM Frameworks for Application Development  
**Syllabus Topic**: 3.4 LlamaIndex framework for document search: Data loaders, Node parsing, VectorStoreIndex, Query & Chat Engines  
**Model Provider**: Google Gemini (`models/text-embedding-004` & `gemini-1.5-flash`) via `llama-index` / First-Principles Simulator  
**Learning Outcome**: Master the 5 stages of the LlamaIndex RAG pipeline (Loading, Splitting, Indexing, Storing, Querying); load local documents using `SimpleDirectoryReader`; chunk documents into relational `Node` objects; build and persist `VectorStoreIndex` to disk; inspect source citations and relevance scores; and construct both stateless Query Engines and stateful multi-turn Chat Engines.

## Part 1: Theoretical Foundations — LlamaIndex & Data-Centric AI

### 1.1 The Need for Retrieval-Augmented Generation (RAG)
Pre-trained Large Language Models (LLMs) suffer from three critical architectural limitations:
1. **Knowledge Cutoff**: Models cannot answer questions about events occurring after their training date.
2. **Lack of Proprietary Knowledge**: Foundation models have zero access to private institutional data, course syllabi, local files, or internal enterprise databases.
3. **Context Window Costs & Latency**: Sending an entire 500-page manual in every prompt exhausts token limits and degrades model focus (*Lost in the Middle* phenomenon).

**RAG (Retrieval-Augmented Generation)** solves this by:
1. Storing external documents in small semantic chunks.
2. Finding only the **top-$k$ most relevant chunks** matching a user's question via vector cosine similarity.
3. Injecting only those relevant chunks into the LLM prompt as context.

### 1.2 Why LlamaIndex?
While LangChain is a general-purpose orchestration framework, **LlamaIndex** is a **data-first framework** specifically optimized for connecting private data sources to LLMs.

```mermaid
graph TD
    subgraph Ingestion & Indexing Pipeline
        RawFiles[Data Files: TXT / PDF / MD] --> Loader[SimpleDirectoryReader]
        Loader --> Docs[Document Objects]
        Docs --> Splitter[SentenceSplitter / NodeParser]
        Splitter --> Nodes[Atomic Node Chunks]
        Nodes --> Embed[text-embedding-004]
        Embed --> Index[VectorStoreIndex]
        Index --> Disk[(StorageContext: ./storage on Disk)]
    end

    subgraph Querying & Synthesis Pipeline
        UserQuery[User Question] --> Retriever[Vector Retriever]
        Disk --> Retriever
        Retriever --> TopNodes[Top-K Relevant Chunks]
        TopNodes --> Synthesizer[Response Synthesizer]
        UserQuery --> Synthesizer
        Synthesizer --> Gemini[Google Gemini LLM]
        Gemini --> Response[Grounded Answer + Source Citations]
    end
```

### 1.3 Core Abstractions in LlamaIndex

| Abstraction | Description | Example in this Practical |
| :--- | :--- | :--- |
| **`Document`** | Container wrapping an entire raw source file with metadata. | `unit-3/data/syllabus_pgdca.txt` |
| **`Node`** | Atomic semantic chunk of a `Document` with relationships to predecessor/successor chunks. | A 256-token paragraph with metadata |
| **`NodeParser`** | Algorithmic splitter dividing Documents into Nodes while respecting sentence boundaries. | `SentenceSplitter(chunk_size=512, chunk_overlap=50)` |
| **`VectorStoreIndex`** | Mathematical index mapping node vector embeddings for fast nearest-neighbor search. | `VectorStoreIndex.from_documents(docs)` |
| **`StorageContext`** | Persistence manager saving vectors and document stores to disk. | `index.storage_context.persist(persist_dir='./storage')` |
| **`QueryEngine`** | High-level interface performing retrieval + response synthesis for single-turn Q&A. | `index.as_query_engine(similarity_top_k=2)` |
| **`ChatEngine`** | Stateful conversational agent with conversation memory over retrieved documents. | `index.as_chat_engine(chat_mode='condense_question')` |
| **`Settings`** | Modern unified global singleton (replaces legacy `ServiceContext`). | `Settings.llm = ...`, `Settings.embed_model = ...` |

## Part 2: Inspecting the Knowledge Base & Environment Setup

Before building an index, let's inspect our local knowledge files located inside `unit-3/data/`.

In [1]:
# Part 2 Code: Inspecting local knowledge base directory and files
import os
from pathlib import Path
from dotenv import load_dotenv

# Load API keys from workspace .env
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

data_dir = Path("data")
print("=" * 60)
print("📂 INSPECTING LOCAL KNOWLEDGE BASE (data/):")
print("=" * 60)

if not data_dir.exists():
    print("⚠️ 'data/' directory not found. Creating sample files...")
    data_dir.mkdir(parents=True, exist_ok=True)
    (data_dir / "llamaindex_info.txt").write_text(
        "LlamaIndex is a data framework for LLM applications to connect private datasets.\n"
        "It enables loading data via SimpleDirectoryReader, parsing documents into Nodes, "
        "indexing them inside a VectorStoreIndex, and querying them using a Query Engine."
    )

data_files = list(data_dir.glob("*.txt"))
for f in data_files:
    content = f.read_text(encoding="utf-8").strip()
    print(f"📄 File: {f.name} ({len(content)} chars)")
    print(f"   Preview: {content[:100]}...")
    print("-" * 60)

print(f"\n🔑 GEMINI_API_KEY Status: {'Configured (' + api_key[:8] + '...)' if api_key else 'Not Found (Using Educational Fallback)'}")

📂 INSPECTING LOCAL KNOWLEDGE BASE (data/):
📄 File: student_faq.txt (245 chars)
   Preview: FAQ 1: The PGDCA practical examination carries 40 marks and theory carries 60 marks.
FAQ 2: Students...
------------------------------------------------------------
📄 File: langchain_info.txt (256 chars)
   Preview: LangChain is a framework for developing applications powered by language models.
It provides standar...
------------------------------------------------------------
📄 File: syllabus_pgdca.txt (418 chars)
   Preview: PGDCA Course Structure and Units:
Unit 1 covers Python ecosystem, environments, variables, data stru...
------------------------------------------------------------
📄 File: llamaindex_info.txt (242 chars)
   Preview: LlamaIndex is a data framework for LLM applications to connect private datasets.
It enables loading ...
------------------------------------------------------------

🔑 GEMINI_API_KEY Status: Configured (AIzaSyBC...)


## Part 3: Ingestion & Node Parsing Concepts

### 3.1 Document vs. Node in Practice
A `Document` is the whole file. A `Node` is a chunk of that document.
Notice why **Chunk Overlap** is critical:
* If sentence *"The exam covers Unit 1 and Unit 2"* is split between two chunks without overlap, neither chunk contains the full relationship.
* Overlap ($10\% - 15\%$) preserves cross-boundary continuity.

In [2]:
# Part 3 Code: Document and Node representation
class SimpleDocument:
    def __init__(self, text, metadata=None):
        self.text = text
        self.metadata = metadata or {}
        self.doc_id = self.metadata.get("file_name", "doc_default")

class SimpleNode:
    def __init__(self, text, doc_id, node_id, metadata=None):
        self.text = text
        self.doc_id = doc_id
        self.node_id = node_id
        self.metadata = metadata or {}

def split_text_into_nodes(documents, chunk_size=150, chunk_overlap=30):
    nodes = []
    for doc in documents:
        text = doc.text
        start = 0
        idx = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk = text[start:end]
            node = SimpleNode(
                text=chunk,
                doc_id=doc.doc_id,
                node_id=f"{doc.doc_id}_chunk_{idx}",
                metadata={**doc.metadata, "start_char": start, "end_char": end}
            )
            nodes.append(node)
            idx += 1
            if end == len(text):
                break
            start += (chunk_size - chunk_overlap)
    return nodes

# Load raw documents
loaded_docs = []
for f in data_files:
    loaded_docs.append(SimpleDocument(text=f.read_text(encoding="utf-8"), metadata={"file_name": f.name}))

parsed_nodes = split_text_into_nodes(loaded_docs, chunk_size=160, chunk_overlap=30)
print(f"✅ Successfully ingested {len(loaded_docs)} Documents and parsed them into {len(parsed_nodes)} Nodes.\n")
for n in parsed_nodes[:3]:
    print(f"🔹 [Node ID: {n.node_id}] from '{n.metadata['file_name']}':")
    print(f"   '{n.text.strip()}'\n")

✅ Successfully ingested 4 Documents and parsed them into 9 Nodes.

🔹 [Node ID: student_faq.txt_chunk_0] from 'student_faq.txt':
   'FAQ 1: The PGDCA practical examination carries 40 marks and theory carries 60 marks.
FAQ 2: Students must submit all unit lab notebooks before the final practic'

🔹 [Node ID: student_faq.txt_chunk_1] from 'student_faq.txt':
   'books before the final practical check.
FAQ 3: The minimum passing grade for each unit practical evaluation is 50%.'

🔹 [Node ID: langchain_info.txt_chunk_0] from 'langchain_info.txt':
   'LangChain is a framework for developing applications powered by language models.
It provides standard, extendable interfaces for Prompts, LLMs, Memory, and Agen'



## Part 4: Production LlamaIndex Implementation (v0.10+ Architecture)

Here is the standard, production-ready LlamaIndex code pattern using modern `Settings`, `SimpleDirectoryReader`, and `VectorStoreIndex`.

> **Note**: If `llama-index` is installed in your Python environment, this cell will execute live with Google Gemini; otherwise, it will cleanly report the setup instructions.

In [3]:
# Part 4 Code: Modern LlamaIndex Setup with Settings & Persistence
try:
    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
    from llama_index.llms.google_genai import GoogleGenAI
    from llama_index.embeddings.google_genai import GoogleGenAIEmbedding   
    print("✅ LlamaIndex Core packages detected!")
    
    # 1. Configure Global Settings

    if api_key:
        Settings.llm=llm = GoogleGenAI(model="gemini-3.5-flash-lite",api_key=api_key)
        Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-2", api_key=api_key)
    Settings.chunk_size = 512
    Settings.chunk_overlap = 50
    
    PERSIST_DIR = "./storage"
    
    # 2. Check if index already exists to avoid redundant embedding costs
    if not os.path.exists(PERSIST_DIR):
        print("📁 Creating new VectorStoreIndex from 'data/' directory...")
        documents = SimpleDirectoryReader("data").load_data()
        index = VectorStoreIndex.from_documents(documents)
        index.storage_context.persist(persist_dir=PERSIST_DIR)
        print(f"💾 Index successfully persisted to '{PERSIST_DIR}'.")
    else:
        print(f"⚡ Loading existing index from '{PERSIST_DIR}' (Zero API cost!)...")
        storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
        index = load_index_from_storage(storage_context)
        
    # 3. Create Query Engine and query
    query_engine = index.as_query_engine(similarity_top_k=2)
    response = query_engine.query("What is LlamaIndex and how does it work?")
    print("=" * 60)
    print("🤖 LLAMAININDEX QUERY ENGINE RESPONSE:")
    print("=" * 60)
    print(response)
    
    # 4. Display Citations & Sources
    print("\n📚 SOURCE CITATIONS:")
    for node in response.source_nodes:
        print(f"- Source File: {node.metadata.get('file_name', 'Unknown')}")
        print(f"  Relevance Score: {node.score}")
        print(f"  Excerpt: {node.text[:120]}...")
        
except ImportError as e:
    print("ℹ️ Note: 'llama-index' package is not installed in current environment.")
    print("   To install modern LlamaIndex with Gemini support, run:")
    print("   !pip install llama-index llama-index-llms-gemini llama-index-embeddings-gemini\n")
    print(f"   (Import message: {e})")
    print("\n👉 Proceed to Part 5 to run the complete, transparent First-Principles LlamaIndex Engine!")

✅ LlamaIndex Core packages detected!
📁 Creating new VectorStoreIndex from 'data/' directory...
💾 Index successfully persisted to './storage'.


RuntimeError: asyncio.run() cannot be called from a running event loop

In [3]:
%pip install llama-index-llms-google-genai llama-index
%pip install llama-index-embeddings-google-genai



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached llama_index_embeddings_google_genai-0.6.0-py3-none-any.whl.metadata (1.1 kB)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install llama-index-embeddings-google-genai

  Using cached aiohttp-3.14.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.3 kB)
  Using cached aiosqlite-0.22.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.5.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pillow-12.3.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
INFO: pip is looking at multiple versions of llama-index-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtr

## Part 5: Complete First-Principles LlamaIndex Simulator (100% Offline & Transparent)

To ensure every student can understand and experiment with the internal mechanics of LlamaIndex even without cloud API credits or network access, we implement the complete **5-Stage Pipeline from First Principles**:
1. **Vector Embedding generation** (semantic representation via character-frequency vector projection).
2. **VectorStoreIndex construction**.
3. **Similarity Search Retrieval** via Cosine Distance.
4. **Response Synthesizer** (combining top chunks + question prompt).
5. **Source Citation Extraction** (attributing file names, scores, and chunk offsets).

In [4]:
# Part 5 Code: First-Principles LlamaIndex Engine
import math
import re

class FirstPrinciplesVectorStoreIndex:
    """
    An educational, self-contained implementation of LlamaIndex's VectorStoreIndex.
    Operates 100% locally with zero external network or SDK dependencies.
    """
    def __init__(self, nodes):
        self.nodes = nodes
        self.vocabulary = self._build_vocabulary([n.text for n in nodes])
        self.embeddings = [self._embed_text(n.text) for n in nodes]
        print(f"📐 Built Index with {len(nodes)} nodes across {len(self.vocabulary)} vocabulary dimensions.")

    def _tokenize(self, text):
        return re.findall(r'\b[a-zA-Z0-9_]{3,}\b', text.lower())

    def _build_vocabulary(self, texts):
        vocab = set()
        for t in texts:
            vocab.update(self._tokenize(t))
        return sorted(list(vocab))

    def _embed_text(self, text):
        tokens = self._tokenize(text)
        counts = {}
        for tok in tokens:
            counts[tok] = counts.get(tok, 0) + 1
        # Vector projection
        vector = [counts.get(word, 0) for word in self.vocabulary]
        # Normalize vector magnitude
        magnitude = math.sqrt(sum(v * v for v in vector))
        if magnitude == 0:
            return [0.0] * len(self.vocabulary)
        return [v / magnitude for v in vector]

    def _cosine_similarity(self, v1, v2):
        return sum(x * y for x, y in zip(v1, v2))

    def as_retriever(self, similarity_top_k=2):
        return FirstPrinciplesRetriever(self, top_k=similarity_top_k)

    def as_query_engine(self, similarity_top_k=2, response_mode="compact"):
        retriever = self.as_retriever(similarity_top_k=similarity_top_k)
        return FirstPrinciplesQueryEngine(retriever, response_mode=response_mode)

    def as_chat_engine(self, chat_mode="condense_question", similarity_top_k=2):
        query_engine = self.as_query_engine(similarity_top_k=similarity_top_k)
        return FirstPrinciplesChatEngine(query_engine, chat_mode=chat_mode)


class FirstPrinciplesRetriever:
    def __init__(self, index, top_k=2):
        self.index = index
        self.top_k = top_k

    def retrieve(self, query_str):
        q_vec = self.index._embed_text(query_str)
        scored_nodes = []
        for node, n_vec in zip(self.index.nodes, self.index.embeddings):
            score = self.index._cosine_similarity(q_vec, n_vec)
            scored_nodes.append((score, node))
        # Sort descending by score
        scored_nodes.sort(key=lambda x: x[0], reverse=True)
        return scored_nodes[:self.top_k]


class FirstPrinciplesResponse:
    def __init__(self, response_text, source_nodes):
        self.response = response_text
        self.source_nodes = source_nodes

    def __str__(self):
        return self.response


class FirstPrinciplesQueryEngine:
    def __init__(self, retriever, response_mode="compact"):
        self.retriever = retriever
        self.response_mode = response_mode

    def query(self, query_str):
        results = self.retriever.retrieve(query_str)
        if not results or results[0][0] < 0.01:
            return FirstPrinciplesResponse(
                response_text="[No relevant context found in documents to answer query.]",
                source_nodes=[]
            )

        # Synthesize answer using compact mode (combining top contexts)
        context_snippets = [f"[{node.metadata['file_name']}]: '{node.text.strip()}'" for score, node in results]
        combined_context = " \n ".join(context_snippets)
        synthesized = (
            f"Based on {len(results)} retrieved source chunks:\n"
            f"{combined_context}\n"
            f"-> Synthesis answering '{query_str}': Successfully resolved using document context."
        )
        return FirstPrinciplesResponse(response_text=synthesized, source_nodes=results)

# Instantiate the educational index
fp_index = FirstPrinciplesVectorStoreIndex(parsed_nodes)
fp_query_engine = fp_index.as_query_engine(similarity_top_k=2)
print("🚀 First-Principles LlamaIndex Query Engine Ready!")

📐 Built Index with 9 nodes across 105 vocabulary dimensions.
🚀 First-Principles LlamaIndex Query Engine Ready!


## Part 6: Executing Queries & Inspecting Citations

A core strength of LlamaIndex is **Citation Transparency**. Every response provides direct access to `source_nodes`, allowing developers and end-users to verify exactly which files and passages grounded the output.

In [5]:
# Part 6 Code: Running test queries with source attribution
queries = [
    "What is LlamaIndex and what does it do?",
    "What topics are covered in the PGDCA Unit 3 syllabus?",
    "How does LangChain differ?"
]

for q in queries:
    print("=" * 70)
    print(f"❓ QUERY: {q}")
    print("=" * 70)
    resp = fp_query_engine.query(q)
    print(resp)
    print("\n📑 SOURCE ATTRIBUTIONS (CITATIONS):")
    for score, node in resp.source_nodes:
        print(f"  • File: {node.metadata['file_name']} | Node ID: {node.node_id} | Similarity: {score:.4f}")
        print(f"    Chunk: \"{node.text.strip()[:100]}...\"")
    print()

❓ QUERY: What is LlamaIndex and what does it do?
Based on 2 retrieved source chunks:
[syllabus_pgdca.txt]: 'chains, memories, LlamaIndex documents search, and vector stores.
Unit 4 covers local LLM runtimes, Ollama, model quantizations, and deploying final systems.' 
 [syllabus_pgdca.txt]: 's Advanced APIs, parameter tunings, concurrency, rate limits, and cost calculation models.
Unit 3 covers LLM Frameworks, LangChain chains, memories, LlamaIndex'
-> Synthesis answering 'What is LlamaIndex and what does it do?': Successfully resolved using document context.

📑 SOURCE ATTRIBUTIONS (CITATIONS):
  • File: syllabus_pgdca.txt | Node ID: syllabus_pgdca.txt_chunk_2 | Similarity: 0.4523
    Chunk: "chains, memories, LlamaIndex documents search, and vector stores.
Unit 4 covers local LLM runtimes, ..."
  • File: syllabus_pgdca.txt | Node ID: syllabus_pgdca.txt_chunk_1 | Similarity: 0.3244
    Chunk: "s Advanced APIs, parameter tunings, concurrency, rate limits, and cost calculation models.
Uni

## Part 7: Building a Conversational Chat Engine (Multi-Turn RAG)

Unlike a single-turn `QueryEngine`, a **`ChatEngine`** maintains conversational history. When a user asks a follow-up question containing pronouns (e.g., *"Does it support memory?"*), the chat engine rewrites the query using prior turns into a standalone search query (*`condense_question`* mode).

In [6]:
# Part 7 Code: Multi-turn Chat Engine with Conversational Memory
class FirstPrinciplesChatEngine:
    def __init__(self, query_engine, chat_mode="condense_question"):
        self.query_engine = query_engine
        self.chat_mode = chat_mode
        self.chat_history = []

    def _condense_question(self, user_msg):
        """Reformulates follow-up queries if user references previous context."""
        if not self.chat_history:
            return user_msg
        last_turn = self.chat_history[-1]
        # If user uses pronouns like 'it', 'they', or 'that', substitute subject from previous turn
        if re.search(r'\b(it|this|that|they)\b', user_msg, re.IGNORECASE):
            condensed = f"{user_msg} (referring to topic in: '{last_turn['user']}')"
            print(f"   🔄 [Query Condensation]: '{user_msg}' -> '{condensed}'")
            return condensed
        return user_msg

    def chat(self, user_msg):
        standalone_query = self._condense_question(user_msg)
        response = self.query_engine.query(standalone_query)
        self.chat_history.append({"user": user_msg, "ai": str(response)})
        return response

    def reset(self):
        self.chat_history.clear()

# Test Multi-Turn Dialogue
chat_engine = fp_index.as_chat_engine(chat_mode="condense_question")

conversation = [
    "Tell me about LlamaIndex.",
    "Does it connect to private datasets?",  # Follow-up with pronoun 'it'
    "What is covered in Unit 3?"
]

print("=" * 70)
print("💬 MULTI-TURN RAG CHAT ENGINE DEMO:")
print("=" * 70)
for turn, msg in enumerate(conversation, 1):
    print(f"\n[Turn {turn}] User: '{msg}'")
    ai_res = chat_engine.chat(msg)
    print(f"[Turn {turn}] Assistant:\n{ai_res}\n")
    print("-" * 50)

💬 MULTI-TURN RAG CHAT ENGINE DEMO:

[Turn 1] User: 'Tell me about LlamaIndex.'
[Turn 1] Assistant:
Based on 2 retrieved source chunks:
[syllabus_pgdca.txt]: 's Advanced APIs, parameter tunings, concurrency, rate limits, and cost calculation models.
Unit 3 covers LLM Frameworks, LangChain chains, memories, LlamaIndex' 
 [llamaindex_info.txt]: 'LlamaIndex is a data framework for LLM applications to connect private datasets.
It enables loading data via SimpleDirectoryReader, parsing documents into Nodes'
-> Synthesis answering 'Tell me about LlamaIndex.': Successfully resolved using document context.

--------------------------------------------------

[Turn 2] User: 'Does it connect to private datasets?'
   🔄 [Query Condensation]: 'Does it connect to private datasets?' -> 'Does it connect to private datasets? (referring to topic in: 'Tell me about LlamaIndex.')'
[Turn 2] Assistant:
Based on 2 retrieved source chunks:
[llamaindex_info.txt]: 'LlamaIndex is a data framework for LLM applicat

## Part 8: Student Hands-On Lab Exercises

### Exercise 1: Ingesting Custom Data
**Task**: Create a new document `unit-3/data/student_faq.txt` with at least 3 FAQ pairs regarding exam rules. Ingest it into your index and verify that questions regarding exam rules retrieve the new document.

In [7]:
# Student Exercise 1 Solution: Creating and Ingesting student_faq.txt
faq_path = Path("data") / "student_faq.txt"
faq_content = (
    "FAQ 1: The PGDCA practical examination carries 40 marks and theory carries 60 marks.\n"
    "FAQ 2: Students must submit all unit lab notebooks before the final practical check.\n"
    "FAQ 3: The minimum passing grade for each unit practical evaluation is 50%."
)
faq_path.write_text(faq_content, encoding="utf-8")
print(f"✅ Wrote new FAQ file: {faq_path}")

# Reload all documents and rebuild index
all_docs = []
for f in Path("data").glob("*.txt"):
    all_docs.append(SimpleDocument(text=f.read_text(encoding="utf-8"), metadata={"file_name": f.name}))

updated_nodes = split_text_into_nodes(all_docs, chunk_size=150, chunk_overlap=30)
updated_index = FirstPrinciplesVectorStoreIndex(updated_nodes)
updated_engine = updated_index.as_query_engine(similarity_top_k=1)

test_faq_query = "What is the minimum passing grade for practicals?"
faq_response = updated_engine.query(test_faq_query)
print(f"\n❓ Question: '{test_faq_query}'")
print(f"💡 Answer:\n{faq_response}")

✅ Wrote new FAQ file: data/student_faq.txt
📐 Built Index with 10 nodes across 105 vocabulary dimensions.

❓ Question: 'What is the minimum passing grade for practicals?'
💡 Answer:
Based on 1 retrieved source chunks:
[student_faq.txt]: 't lab notebooks before the final practical check.
FAQ 3: The minimum passing grade for each unit practical evaluation is 50%.'
-> Synthesis answering 'What is the minimum passing grade for practicals?': Successfully resolved using document context.


### Exercise 2: Implementing a Relevance Score Threshold
**Task**: In production, returning irrelevant documents when cosine similarity is low causes hallucinations. Implement a minimum threshold (e.g. `threshold = 0.20`) that rejects queries when no documents meet the quality bar.

In [8]:
# Student Exercise 2 Solution: Guarded Query Engine with Similarity Threshold
def guarded_query(engine, query_str, threshold=0.15):
    results = engine.retriever.retrieve(query_str)
    if not results or results[0][0] < threshold:
        top_score = results[0][0] if results else 0.0
        return f"⚠️ Guardrail Triggered: Top match score ({top_score:.4f}) is below threshold ({threshold}). Query rejected to prevent hallucination."
    return engine.query(query_str)

# Test with in-domain query
print("Query 1 (In Domain): 'What is LlamaIndex?')")
print(guarded_query(updated_engine, "What is LlamaIndex?", threshold=0.15))
print("\n" + "="*60 + "\n")

# Test with out-of-domain query
print("Query 2 (Out of Domain): 'How do I bake chocolate cake?')")
print(guarded_query(updated_engine, "How do I bake chocolate cake?", threshold=0.15))

Query 1 (In Domain): 'What is LlamaIndex?')
Based on 1 retrieved source chunks:
[syllabus_pgdca.txt]: 'rameworks, LangChain chains, memories, LlamaIndex documents search, and vector stores.
Unit 4 covers local LLM runtimes, Ollama, model quantizations,'
-> Synthesis answering 'What is LlamaIndex?': Successfully resolved using document context.


Query 2 (Out of Domain): 'How do I bake chocolate cake?')
⚠️ Guardrail Triggered: Top match score (0.0000) is below threshold (0.15). Query rejected to prevent hallucination.


## Summary & Key Takeaways

1. **RAG Architecture**: Bridges the gap between static LLM foundation models and dynamic private data without retraining weights.
2. **The 5 Stages**: Ingestion (`SimpleDirectoryReader`), Node Parsing (`SentenceSplitter`), Indexing (`VectorStoreIndex`), Storage (`StorageContext`), and Querying (`QueryEngine` / `ChatEngine`).
3. **Document vs. Node**: Documents wrap files; Nodes are the atomic relational building blocks that preserve parent, preceding, and succeeding chunk links.
4. **Chunk Overlap**: Eliminates information loss at sentence boundaries.
5. **Storage Persistence**: Saves embeddings to `./storage` to avoid costly re-embedding calls on every execution.
6. **Query Engines vs. Chat Engines**: Query engines provide stateless one-shot retrieval; Chat engines provide multi-turn conversation memory with query condensation.
7. **Citations & Auditing**: Always verify `response.source_nodes` for production grounding and regulatory compliance.

---